<a href="https://colab.research.google.com/github/carrisian/bigdata-predictive-model/blob/main/notebooks/sat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ⚡ Sistema de Alerta Temprana (S.A.T.)

El objetivo de este bloque es realizar una **inferencia en tiempo real** utilizando el modelo de Deep Learning entrenado (redes LSTM) para predecir concentraciones de PM10 y validar, mediante lógica física, la ocurrencia de eventos de intrusión sahariana.

### 📝 Preparación de los datos (Ventana temporal)
El modelo requiere una secuencia de 24 horas consecutivas para ser preciso. Estas se introducen mediante **8 bloques de 3 horas cada uno**.

*   **Instrucciones:** Introduce los **16 valores** separados por comas siguiendo el orden indicado.
*   **Orden de las variables:** PM10, PM2.5, NO2, Ozono, CO, SO2, Temp, Hum, Viento_Vel, Viento_Dir, Presion, Nubes, UV_Real, UV_Cielo_Despejado, Lluvia, Radiacion.
*   **Nomenclatura temporal:**
    *   `h_24`: Datos correspondientes a **24 horas antes** del momento de la predicción.
    *   `h_21`: Datos correspondientes a **21 horas antes**.
    *   ... y así sucesivamente hasta `h_3` (3 horas antes).

### ¿Cómo interpretar los resultados?

Para obtener una predicción fiable, el sistema coteja la **predicción algorítmica** con nuestro **Índice Sintético de Intrusión (ISI)**:

- **Predicción IA (PM10):** Es la estimación de la carga de partículas. Esta cifra no es un valor absoluto, sino una proyección basada en la dinámica observada en los datos históricos.
- **Score Físico (ISI):** Es nuestra "validación de realidad". Evalúa 4 pilares físicos (masa, aridez, atenuación radiativa y dirección del viento).
    - **Score bajo (0-2):** El sistema detecta condiciones atmosféricas normales.
    - **Score alto (3-4):** ** Existe una ** convergencia física ** de variables que coinciden con el comportamiento típico de una intrusión sahariana.



In [ ]:
#@title Sistema de Alerta Temprana { display-mode: "form" }
# --- 1. CONFIGURACIÓN E INICIALIZACIÓN ---
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import logging
from google.colab import drive
from datetime import datetime

# Silenciar avisos técnicos de TensorFlow y logs innecesarios
tf.get_logger().setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Montar Drive
if not os.path.isdir('/content/drive'): drive.mount('/content/drive')

PATH_BASE = "/content/drive/MyDrive/TFM_Profesorado/Modelos_IA_Listos/"
RUTA_MODELO = os.path.join(PATH_BASE, "modelo_GLOBAL.keras")

# Carga del modelo (sin avisos)
model = tf.keras.models.load_model(RUTA_MODELO, compile=False)

# --- 2. 📝 ENTRADA DE DATOS (24h en 8 bloques de 3h) ---
#@markdown ### Instrucciones: Introduce los 16 valores separados por comas.
#@markdown **Orden:** PM10, PM2.5, NO2, Ozono, CO, SO2, Temp, Hum, Viento_Vel, Viento_Dir, Presion, Nubes, UV_Real, UV_Cielo_Despejado, Lluvia, Radiacion
h_24 = "40, 20, 15, 60, 0.5, 5, 22, 50, 5, 180, 1013, 2, 4.5, 6.0, 0, 200" #@param {type:"string"}
h_21 = "45, 22, 16, 58, 0.6, 5, 21, 45, 6, 175, 1012, 2, 3.0, 5.0, 0, 150" #@param {type:"string"}
h_18 = "50, 25, 18, 55, 0.7, 6, 20, 40, 7, 190, 1011, 3, 1.5, 3.0, 0, 100" #@param {type:"string"}
h_15 = "55, 38, 20, 50, 0.8, 6, 19, 38, 8, 200, 1010, 3, 0.5, 1.5, 0, 50"  #@param {type:"string"}
h_12 = "62, 35, 22, 45, 0.9, 7, 18, 35, 9, 210, 1009, 4, 0.2, 0.5, 0, 20"  #@param {type:"string"}
h_9  = "68, 30, 25, 40, 1.0, 7, 17, 30, 10, 215, 1008, 4, 0.1, 0.2, 0, 10" #@param {type:"string"}
h_6  = "72, 28, 28, 35, 1.1, 8, 16, 28, 11, 220, 1007, 5, 0.0, 0.0, 0, 0"  #@param {type:"string"}
h_3  = "75, 25, 30, 30, 1.2, 8, 15, 25, 12, 225, 1006, 5, 0.0, 0.0, 0, 0"  #@param {type:"string"}

# --- 3. PROCESAMIENTO E INFERENCIA ---
entradas = [h_24, h_21, h_18, h_15, h_12, h_9, h_6, h_3]
data_matrix = np.array([[float(x) for x in h.split(',')] for h in entradas])

# Reshape (1, 8, 16) para el modelo LSTM
input_ia = data_matrix.reshape(1, 8, 16).astype(np.float32)
prediccion_ia = model.predict(input_ia, verbose=0)[0][0]

# --- 4. CÁLCULO E INFORME FINAL ---
# Score ISI basado en las 4 variables clave de tu investigación
score_isi = (data_matrix[:,0].mean() > 60) + (data_matrix[:,1].mean() < 40) + \
            (data_matrix[:,2].mean() > 1.5) + (135 <= data_matrix[:,3].mean() <= 225)

print(f"\n--- ⚡ RESULTADO DEL S.A.T. ---")
print(f"Predicción IA (PM10): {prediccion_ia:.2f} µg/m³")
print(f"Score Físico ISI: {int(score_isi)}/4")
estado = "🚩 ALERTA ROJA (Intrusión Confirmada)" if score_isi >= 3 else "✅ ESTADO NORMAL"
print(f"Estado: {estado}")

# Guardado histórico en Drive
logs = pd.DataFrame([{"Fecha": datetime.now(), "IA_PM10": prediccion_ia, "Score": score_isi}])
logs.to_csv(os.path.join(PATH_BASE, "historial_sat.csv"), mode='a', header=not os.path.exists(os.path.join(PATH_BASE, "historial_sat.csv")), index=False)
print(f"💾 Informe guardado en: historial_sat.csv")


--- ⚡ RESULTADO DEL S.A.T. ---
Predicción IA (PM10): 0.71 µg/m³
Score Físico ISI: 1/4
Estado: ✅ ESTADO NORMAL
💾 Informe guardado en: historial_sat.csv
